# Zebrafish heart rate from high-speed video

University of Sheffield — Year 3 Physics, Industrial Group Project

**Aim:** measure heart rate from videos of freely swimming zebrafish larvae and compare the results with manual beat counts.

The notebook contains the training work, analysis of individual recordings and the batch analysis. Run the relevant sections rather than Run all: training and pose estimation may need different Colab environments. The simulated checks do not use experimental data.

## 1. Video acquisition and inspection

In [ ]:
%pip -q install opencv-python-headless scikit-image numpy pandas scipy matplotlib pyyaml
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import filters, morphology, measure
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/ZebrafishHeartRate')
project = ROOT / 'detectron2'
project.mkdir(parents=True, exist_ok=True)
VIDEO_DIR = ROOT / 'videos'
OUT = ROOT / 'outputs'
FRAMES = OUT / 'label_candidates'
for directory in (VIDEO_DIR, OUT, FRAMES):
    directory.mkdir(parents=True, exist_ok=True)

VIDEO_NAME = None   # Set to a filename to select a particular video
FPS_OVERRIDE = None # Use camera acquisition metadata if container FPS is incorrect
SAMPLE_COUNT = 80
MIN_AREA = 30
MAX_AREA_FRACTION = 0.5
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv'}


In [ ]:
videos = sorted(p for p in VIDEO_DIR.iterdir() if p.suffix.lower() in VIDEO_EXTENSIONS)
if not videos:
    raise FileNotFoundError(f'Add a video to {VIDEO_DIR}')
video_path = VIDEO_DIR / VIDEO_NAME if VIDEO_NAME else videos[0]
if not video_path.exists():
    raise FileNotFoundError(video_path)
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f'Cannot open {video_path}')
reported_fps = cap.get(cv2.CAP_PROP_FPS)
fps = float(FPS_OVERRIDE if FPS_OVERRIDE is not None else reported_fps)
count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if fps <= 0 or count < 1:
    raise ValueError('Check the frame rate and frame count against acquisition metadata')
print(pd.Series({'file': video_path.name, 'fps': fps, 'container_fps': reported_fps,
                 'frames': count, 'width': w, 'height': h, 'duration_s': count / fps}))

def read_frame(index):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Cannot decode frame {index}')
    return frame

sample_indices = np.unique(np.linspace(0, count - 1, min(count, SAMPLE_COUNT), dtype=int))
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, index in zip(axes, np.linspace(0, len(sample_indices)-1, 4, dtype=int)):
    frame_number = int(sample_indices[index])
    ax.imshow(cv2.cvtColor(read_frame(frame_number), cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {frame_number}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 2. Background estimation and baseline foreground segmentation

In [ ]:
# Median background estimate. Assumes a fixed camera and moving foreground.
gray_samples = [cv2.cvtColor(read_frame(i), cv2.COLOR_BGR2GRAY) for i in sample_indices]
background = np.median(np.stack(gray_samples), axis=0).astype(np.uint8)
cv2.imwrite(str(OUT / f'{video_path.stem}_background.png'), background)

def segment_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    difference = cv2.absdiff(gray, background)
    if difference.min() == difference.max():
        return np.zeros_like(gray, dtype=bool), difference
    cutoff = filters.threshold_otsu(difference)
    binary = difference > cutoff
    binary = morphology.remove_small_objects(binary, min_size=MIN_AREA)
    binary = morphology.binary_closing(binary, morphology.disk(2))
    return binary, difference

frame = read_frame(int(sample_indices[len(sample_indices)//2]))
mask, difference = segment_frame(frame)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, data, title in zip(axes, [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), difference, mask],
                           ['Raw frame', 'Background difference', 'Foreground mask']):
    ax.imshow(data, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Baseline segmentation: one candidate component per frame.
# Candidate selection by area is not a validated fish detector.
cap = cv2.VideoCapture(str(video_path))
rows = []
frame_number = 0
max_area = w * h * MAX_AREA_FRACTION
while True:
    ok, frame = cap.read()
    if not ok:
        break
    mask, _ = segment_frame(frame)
    regions = [r for r in measure.regionprops(measure.label(mask))
               if MIN_AREA <= r.area <= max_area]
    if regions:
        region = max(regions, key=lambda r: r.area)
        y0, x0, y1, x1 = region.bbox
        rows.append({'frame': frame_number, 'time_s': frame_number/fps,
                     'x': float(region.centroid[1]), 'y': float(region.centroid[0]),
                     'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1,
                     'area_px': int(region.area)})
    frame_number += 1
cap.release()
columns = ['frame', 'time_s', 'x', 'y', 'x0', 'y0', 'x1', 'y1', 'area_px']
detections = pd.DataFrame(rows, columns=columns)
detection_path = OUT / f'{video_path.stem}_fish_candidates.csv'
detections.to_csv(detection_path, index=False)
print(f'{len(detections)} candidate frames / {frame_number} decoded frames')
print(detection_path)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, i in zip(axes.flat, np.linspace(0, frame_number - 1, 6, dtype=int)):
    frame = read_frame(i)
    candidate = detections.loc[detections.frame == i]
    if not candidate.empty:
        r = candidate.iloc[0]
        cv2.rectangle(frame, (int(r.x0), int(r.y0)), (int(r.x1), int(r.y1)), (0, 255, 0), 2)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {i}')
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Export diverse frames for annotation, including frames without a detection.
# A group member should review and correct the selection before labelling.
annotation_indices = np.unique(np.linspace(0, frame_number - 1,
                                           min(100, frame_number), dtype=int))
for i in annotation_indices:
    cv2.imwrite(str(FRAMES / f'{video_path.stem}_{i:07d}.png'), read_frame(i))
print(f'Exported {len(annotation_indices)} candidate annotation frames to {FRAMES}')


## 3. Fish segmentation (Detectron2)

Mask R-CNN using the existing set of 65 images and MakeSense.ai COCO polygons. Keep frames from the same recording together when splitting the data. The background masks in section 2 are for comparison, not training labels.

### Install Detectron2

In [ ]:
import sys, subprocess, torch
print('Python:',sys.version)
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.is_available())
# Installation can take a few minutes in Colab.
!python -m pip install -q opencv-python-headless pycocotools matplotlib pandas pillow pyyaml scikit-image scipy
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
import detectron2
print('Detectron2 imported successfully')

### Image folders and annotations

Extract `zebrafish_65_unique_images.zip`. In MakeSense.ai, label the fish using polygons and export COCO JSON. Only label the heart if it can be identified reliably.

```text
MyDrive/ZebrafishHeartRate/detectron2/
    train/images/
    train/labels.json
    validate/images/
    validate/labels.json
    inference_images/
    Model/
```

Check whether the 65 images come from separate recordings before splitting. A split of neighbouring frames can be used to check that training works, but not to estimate generalisation to new fish.

### Optional: extract image ZIP to Drive

In [ ]:
from google.colab import files
from zipfile import ZipFile
uploaded = files.upload()  # select zebrafish_65_unique_images.zip
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')),None)
if zip_name:
    with ZipFile(zip_name) as z:
        names=[n for n in z.namelist() if n.startswith('images/') and n.endswith('.png')]
        destination=project/'unlabelled_images'
        destination.mkdir(exist_ok=True)
        for name in names:
            (destination/Path(name).name).write_bytes(z.read(name))
    print('Extracted',len(names),'PNG frames to',destination)
else:
    print('No ZIP uploaded')

### Register train/validation datasets

In [ ]:
from detectron2.data import MetadataCatalog, DatasetCatalog, datasets
from pathlib import Path
import json
data_path = project
train_annotations = data_path / 'train' / 'labels.json'
train_images = data_path / 'train' / 'images'
val_annotations = data_path / 'validate' / 'labels.json'
val_images = data_path / 'validate' / 'images'
for annotations, images in [(train_annotations, train_images), (val_annotations, val_images)]:
    assert annotations.exists(), f'Missing COCO annotations: {annotations}'
    assert images.exists(), f'Missing image directory: {images}'
    obj = json.loads(annotations.read_text())
    assert obj.get('images') and obj.get('annotations'), f'No images/annotations in {annotations}'
    assert all((a.get('segmentation') for a in obj['annotations'])), 'Polygon segmentation missing; export polygon COCO annotations'
    missing = [i['file_name'] for i in obj['images'] if not (images / i['file_name']).exists()]
    assert not missing, f'Image names do not match COCO JSON: {missing[:5]}'
    print(annotations, 'images:', len(obj['images']), 'annotations:', len(obj['annotations']))
for catalog in (DatasetCatalog, MetadataCatalog):
    for name in ('my_dataset_train', 'my_dataset_val'):
        if name in catalog:
            catalog.remove(name)
datasets.register_coco_instances('my_dataset_train', {}, str(train_annotations), str(train_images))
datasets.register_coco_instances('my_dataset_val', {}, str(val_annotations), str(val_images))
train_metadata = MetadataCatalog.get('my_dataset_train')
val_metadata = MetadataCatalog.get('my_dataset_val')
train_dataset_dicts = DatasetCatalog.get('my_dataset_train')
val_dataset_dicts = DatasetCatalog.get('my_dataset_val')
assert train_metadata.thing_classes == val_metadata.thing_classes, 'Train/val class names differ'
print('Classes:', train_metadata.thing_classes)


### Visualise datasets

In [ ]:
import matplotlib.pyplot as plt
import random, cv2
from detectron2.utils.visualizer import Visualizer
num_samples_to_view = 3
for d in random.sample(train_dataset_dicts, min(num_samples_to_view, len(train_dataset_dicts))):
    img = cv2.imread(d['file_name'])
    visualizer = Visualizer(img[:, :, ::-1], metadata=train_metadata, scale=0.7)
    vis = visualizer.draw_dataset_dict(d)
    plt.figure(figsize=(9, 7))
    plt.imshow(vis.get_image())
    plt.axis('off')
    plt.show()


# Training
### Set training parameters

In [ ]:
import os,yaml
from detectron2.config import get_cfg
from detectron2 import model_zoo
cfg=get_cfg()
model_directory=project/'Model'
model_directory.mkdir(exist_ok=True)
cfg.OUTPUT_DIR=str(model_directory)
pretrained_model='COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
cfg.merge_from_file(model_zoo.get_config_file(pretrained_model))
cfg.MODEL.WEIGHTS=model_zoo.get_checkpoint_url(pretrained_model)
cfg.DATASETS.TRAIN=('my_dataset_train',)
cfg.DATASETS.TEST=('my_dataset_val',)  # independent validation preferred
cfg.DATALOADER.NUM_WORKERS=2
cfg.SOLVER.IMS_PER_BATCH=2
cfg.SOLVER.BASE_LR=0.00025
cfg.SOLVER.MAX_ITER=300 # initial training run; revisit with more labelled images
cfg.SOLVER.STEPS=[]
cfg.TEST.EVAL_PERIOD=100
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE=128
cfg.MODEL.ROI_HEADS.NUM_CLASSES=len(train_metadata.thing_classes)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST=0.5
cfg.INPUT.MIN_SIZE_TRAIN=(640,800)
cfg.INPUT.MAX_SIZE_TRAIN=1024
cfg.INPUT.MIN_SIZE_TEST=800
cfg.INPUT.MAX_SIZE_TEST=1024
cfg.MODEL.DEVICE='cuda' if __import__('torch').cuda.is_available() else 'cpu'
with open(model_directory/'cfg.yaml','w') as f:f.write(cfg.dump())
metrics_path=model_directory/'metrics.json'
if metrics_path.exists():metrics_path.unlink()
print('Classes:',cfg.MODEL.ROI_HEADS.NUM_CLASSES,'Device:',cfg.MODEL.DEVICE)

### Create Trainer

In [ ]:
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator

class MyTrainer(DefaultTrainer):

    @classmethod
    def build_evaluator(cls, cfg, dataset_name):
        return COCOEvaluator(dataset_name, output_dir=str(Path(cfg.OUTPUT_DIR) / 'eval' / dataset_name))
trainer = MyTrainer(cfg)
trainer.resume_or_load(resume=False)


### Run Training

In [ ]:
trainer.train()
print('Model directory:',model_directory)

### Plot training

In [ ]:
import json, matplotlib.pyplot as plt
results = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axs, ('segm/AP', 'bbox/AP')):
    for name in ('my_dataset_val',):
        points = [(r['iteration'], r[name + '/' + metric]) for r in results if name + '/' + metric in r]
        if points:
            ax.plot(*zip(*points), label=name)
    ax.set(xlabel='Iteration', ylabel=metric)
    ax.grid(True)
    ax.legend() if ax.lines else None
plt.tight_layout()
plt.show()
print('These AP values are exploratory if train/val are from the same recording.')


# Validation and inference
### Loading a trained model from file

In [ ]:
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
cfg = get_cfg()
cfg.merge_from_file(str(model_directory / 'cfg.yaml'))
cfg.MODEL.WEIGHTS = str(model_directory / 'model_final.pth')
assert Path(cfg.MODEL.WEIGHTS).exists(), 'Train the model before running inference'
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor = DefaultPredictor(cfg)


### Run evaluation

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
evaluator = COCOEvaluator('my_dataset_val', output_dir=str(model_directory / 'final_eval'))
val_loader = build_detection_test_loader(cfg, 'my_dataset_val')
print(inference_on_dataset(predictor.model, val_loader, evaluator))


### View results on validation dataset

In [ ]:
from detectron2.utils.visualizer import Visualizer
for d in random.sample(val_dataset_dicts, min(3, len(val_dataset_dicts))):
    img = cv2.imread(d['file_name'])
    outputs = predictor(img)
    v = Visualizer(img[:, :, ::-1], metadata=val_metadata, scale=0.7)
    vis = v.draw_instance_predictions(outputs['instances'].to('cpu'))
    plt.figure(figsize=(9, 7))
    plt.imshow(vis.get_image())
    plt.axis('off')
    plt.show()


### Load and segment an image

`outputs['instances'].pred_masks` holds predicted instance masks; `pred_classes`, `scores`, and `pred_boxes` hold classes, confidence and boxes.

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
inference_dir = project / 'inference_images'
output_dir = project / 'predictions'
output_dir.mkdir(exist_ok=True)
image_paths = sorted(inference_dir.glob('*.png')) if inference_dir.exists() else []
if not image_paths:
    image_paths = [Path(d['file_name']) for d in val_dataset_dicts[:5]]
records = []
for path in image_paths:
    im = cv2.imread(str(path))
    outputs = predictor(im)
    instances = outputs['instances'].to('cpu')
    for j, mask in enumerate(instances.pred_masks.numpy()):
        cls = int(instances.pred_classes[j])
        score = float(instances.scores[j])
        mask_file = output_dir / f'{path.stem}_instance{j:02d}.png'
        cv2.imwrite(str(mask_file), mask.astype('uint8') * 255)
        ys, xs = np.where(mask)
        records.append({'image': path.name, 'class': train_metadata.thing_classes[cls], 'confidence': score, 'centroid_x': float(xs.mean()) if len(xs) else np.nan, 'centroid_y': float(ys.mean()) if len(ys) else np.nan, 'mask_pixels': int(mask.sum()), 'mask_file': mask_file.name})
    print(path.name, 'predicted instances:', len(instances))
pd.DataFrame(records, columns=['image', 'class', 'confidence', 'centroid_x', 'centroid_y', 'mask_pixels', 'mask_file']).to_csv(output_dir / 'predictions.csv', index=False)
print('Saved:', output_dir)


### From images to video

The centroids from these still images are not enough to calculate BPM. For that we need consecutive frames, acquisition FPS and a localised heart.

## 4. Locating the heart (DeepLabCut)

Label `heart_center`, `head` and `tail_base` manually on representative frames. The fish masks are useful for choosing frames but do not provide heart labels. The code uses the DeepLabCut 2.x API; check the environment before training.

In [ ]:
# Install a compatible DeepLabCut 2.x release in a separate Colab runtime if needed.
# Check the current DeepLabCut installation documentation before choosing versions.
# Stage 3 Detectron2 and Stage 4 DeepLabCut may require different dependency environments.
DLC_CONFIG = None
CREATE_DLC_PROJECT = False
if CREATE_DLC_PROJECT:
    import deeplabcut
    DLC_CONFIG = deeplabcut.create_new_project(
        'ZebrafishCardiacLandmarks', 'ResearchGroup', [str(video_path)],
        working_directory=str(ROOT / 'deeplabcut'), copy_videos=False)
    print(DLC_CONFIG)
    print('Set bodyparts in config.yaml and annotate representative frames before training.')

In [ ]:
RUN_DLC_TRAINING = False
if RUN_DLC_TRAINING:
    if DLC_CONFIG is None:
        raise ValueError('Point DLC_CONFIG to an existing labelled DeepLabCut config.yaml')
    import deeplabcut
    deeplabcut.check_labels(DLC_CONFIG)
    deeplabcut.create_training_dataset(DLC_CONFIG)
    deeplabcut.train_network(DLC_CONFIG)
    deeplabcut.evaluate_network(DLC_CONFIG)
    deeplabcut.analyze_videos(DLC_CONFIG, [str(video_path)])

### Checks before analysing video

Check acquisition FPS, exposure, annotation overlays and landmark errors on separate recordings. Save the original checkpoints before retraining.

## 5. Cardiac ROI registration

Single larva, original video coordinates. Use the head and tail-base landmarks to estimate body orientation and the heart landmark for translation. This corrects in-plane rigid motion only; out-of-plane rotation, occlusion and non-rigid deformation require review. No temporal intensity filtering or frame interpolation is applied. Keep rejected frames in the timeline.

In [ ]:
# Can run independently of the Detectron2 training runtime.
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/ZebrafishHeartRate')
VIDEO = ROOT / 'videos' / 'recording.avi'  # change to the analysed recording
LANDMARKS = ROOT / 'recordingDLC.h5'     # matching DeepLabCut output
FPS_OVERRIDE = None
ROI_SIZE = 96
MIN_LIKELIHOOD = 0.90
MIN_AXIS_PX = 10
OUT = ROOT / 'outputs' / VIDEO.stem / 'registration'
OUT.mkdir(parents=True, exist_ok=True)

if LANDMARKS.suffix == '.h5':
    pose = pd.read_hdf(LANDMARKS)
else:
    pose = pd.read_csv(LANDMARKS, header=[0, 1, 2], index_col=0)
if pose.columns.nlevels != 3:
    raise ValueError('Expected single-animal DLC output with scorer/bodyparts/coords')
if len(pose.columns.get_level_values(0).unique()) != 1:
    raise ValueError('Select one scorer before registration')
pose = pose.droplevel(0, axis=1)
parts = ['heart_center', 'head', 'tail_base']
points = {p: pose[p][['x', 'y', 'likelihood']].to_numpy(float) for p in parts}
if not np.array_equal(np.asarray(pose.index, dtype=int), np.arange(len(pose))):
    raise ValueError('Landmark rows must correspond to consecutive original video frames')


In [ ]:
def register_heart(frame, heart, head, tail):
    landmarks = np.stack([heart, head, tail])
    if not np.isfinite(landmarks).all() or landmarks[:, 2].min() < MIN_LIKELIHOOD:
        return None, None, 'low_confidence'
    h, w = frame.shape[:2]
    if np.any(landmarks[:, :2] < 0) or np.any(landmarks[:, 0] >= w) or np.any(landmarks[:, 1] >= h):
        return None, None, 'outside_frame'
    axis = head[:2] - tail[:2]
    if np.linalg.norm(axis) < MIN_AXIS_PX:
        return None, None, 'short_body_axis'
    angle = np.degrees(np.arctan2(axis[1], axis[0]))
    centre = (ROI_SIZE - 1) / 2
    matrix = cv2.getRotationMatrix2D(tuple(heart[:2]), angle, 1.0)
    matrix[:, 2] += centre - heart[:2]
    size = (ROI_SIZE, ROI_SIZE)
    coverage = cv2.warpAffine(np.ones((h, w), np.uint8), matrix, size,
                              flags=cv2.INTER_NEAREST)
    if not coverage.all():
        return None, matrix, 'clipped_roi'
    roi = cv2.warpAffine(frame, matrix, size, flags=cv2.INTER_LINEAR)
    return roi, matrix, 'ok'

# Check transform direction on synthetic landmarks.
test = np.zeros((200, 200), np.uint8)
roi, m, status = register_heart(test, np.array([100.,100.,1.]),
                               np.array([100.,130.,1.]), np.array([100.,70.,1.]))
assert status == 'ok'
assert np.allclose(m @ [100,100,1], [(ROI_SIZE-1)/2]*2)
assert (m @ [100,130,1])[0] > (ROI_SIZE-1)/2


In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
if not cap.isOpened():
    raise FileNotFoundError(VIDEO)
fps = float(FPS_OVERRIDE or cap.get(cv2.CAP_PROP_FPS))
if not np.isfinite(fps) or fps <= 0:
    cap.release()
    raise ValueError('Set acquisition FPS')
rows = []
# Lossless array on disk; rejected frames are zero-filled and explicitly flagged.
rois = np.lib.format.open_memmap(OUT / 'cardiac_rois.npy', mode='w+',
                                dtype=np.uint8, shape=(len(pose), ROI_SIZE, ROI_SIZE))
try:
    for i in range(len(pose)):
        ok, frame = cap.read()
        if not ok:
            raise RuntimeError(f'Video ended before landmark row {i}')
        heart, head, tail = (points[p][i] for p in parts)
        roi, matrix, status = register_heart(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), heart, head, tail)
        rois[i] = roi if roi is not None else 0
        record = dict(frame=i, time_s=i/fps, heart_x=heart[0], heart_y=heart[1],
                      heart_likelihood=heart[2], head_likelihood=head[2],
                      tail_likelihood=tail[2], valid=status == 'ok', status=status)
        record.update({f'm{j}{k}': matrix[j,k] if matrix is not None else np.nan
                       for j in range(2) for k in range(3)})
        rows.append(record)
    if cap.read()[0]:
        raise ValueError('Video contains more frames than the landmark table')
finally:
    cap.release()
    rois.flush()
qc = pd.DataFrame(rows)
qc.to_csv(OUT / 'registration_qc.csv', index=False)
print(qc.status.value_counts())
print(f'Accepted frames: {qc.valid.mean():.1%}')


In [ ]:
valid = qc.loc[qc.valid, 'frame'].to_numpy(int)
if len(valid):
    examples = valid[np.linspace(0, len(valid) - 1, min(6, len(valid)), dtype=int)]
    fig, axes = plt.subplots(1, len(examples), figsize=(3 * len(examples), 3), squeeze=False)
    for ax, i in zip(axes.flat, examples):
        ax.imshow(rois[i], cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'Frame {i}')
        ax.axis('off')
    plt.show()
else:
    print('No accepted frames; inspect landmark predictions.')
plt.figure(figsize=(10, 3))
plt.plot(qc.time_s, qc.heart_likelihood, label='Heart likelihood')
plt.plot(qc.time_s, qc.valid.astype(int), label='Accepted', alpha=0.6)
plt.xlabel('Time (s)')
plt.legend()
plt.show()


Check consecutive crops for heart visibility and movement of nearby tissue. The confidence limit and crop size still need testing. Rejected crops are zero-filled for storage but excluded from signal extraction. Landmark coordinates must refer to the original frame.

## 6. Cardiac signal extraction

Use the registered grayscale crops and registration flags. Central circular analysis mask; inspect its coverage before processing. Keep gaps in the original timeline and filter each contiguous valid segment independently. The frequency band is exploratory and must be checked against the acquisition conditions.

In [ ]:
from scipy import signal
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

REG = OUT  # Stage 5 registration directory
SIGNALS = REG.parent / 'signals'
SIGNALS.mkdir(exist_ok=True)
rois = np.load(REG / 'cardiac_rois.npy', mmap_mode='r')
qc = pd.read_csv(REG / 'registration_qc.csv')
t = qc.time_s.to_numpy(float)
assert len(t) == len(rois) and len(t) > 1
assert np.array_equal(qc.frame.to_numpy(), np.arange(len(t)))
dt = np.diff(t)
assert np.all(dt > 0) and np.allclose(dt, np.median(dt), rtol=1e-5, atol=1e-9)
fps = 1 / np.median(dt)
valid = qc.valid.astype(str).str.lower().eq('true').to_numpy()
LOW_HZ, HIGH_HZ = 0.8, 6.0  # provisional, not physiological exclusion criteria
RADIUS = 20
MIN_SEGMENT_S = 5
if not 0 < LOW_HZ < HIGH_HZ < fps/2:
    raise ValueError('Check acquisition FPS and filter band')
h, w = rois.shape[1:]
y, x = np.ogrid[:h, :w]
mask = (x-(w-1)/2)**2 + (y-(h-1)/2)**2 <= RADIUS**2
assert mask.any() and RADIUS < min(h,w)/2
if valid.any():
    plt.imshow(rois[np.flatnonzero(valid)[0]], cmap='gray')
    plt.contour(mask, levels=[0.5], colors='yellow')
    plt.title('Analysis region — check cardiac coverage')
    plt.show()

In [ ]:
def extract_signals(images, accepted, region, fps):
    values = np.full((len(images), 3), np.nan)
    for i in range(len(images)):
        if not accepted[i]:
            continue
        current = np.asarray(images[i])
        values[i, 0] = current[region].mean()
        if i == 0 or not accepted[i-1]:
            continue
        previous = np.asarray(images[i-1])
        values[i, 1] = cv2.absdiff(previous, current)[region].mean()
        flow = cv2.calcOpticalFlowFarneback(previous, current, None,
                                             0.5, 3, 15, 3, 5, 1.2, 0)
        values[i, 2] = np.linalg.norm(flow[region], axis=1).mean() * fps
    return values

names = ['intensity', 'frame_difference', 'flow_px_s']
raw = extract_signals(rois, valid, mask, fps)
sos = signal.butter(3, [LOW_HZ, HIGH_HZ], btype='bandpass', fs=fps, output='sos')

def filter_segments(values, sos, min_samples):
    result = np.full_like(values, np.nan, dtype=float)
    edges = np.flatnonzero(np.diff(np.r_[False, np.isfinite(values), False]))
    for start, stop in edges.reshape(-1, 2):
        if stop-start < min_samples:
            continue
        try:
            result[start:stop] = signal.sosfiltfilt(sos, values[start:stop])
        except ValueError:
            continue  # segment shorter than filter padding
    return result

results = qc[['frame', 'time_s', 'valid']].copy()
for j, name in enumerate(names):
    results[name] = raw[:, j]
    results[name+'_filtered'] = filter_segments(raw[:,j], sos, int(np.ceil(MIN_SEGMENT_S*fps)))
results.to_csv(SIGNALS / 'cardiac_signals.csv', index=False)
print(results.notna().sum())

In [ ]:
for name in names:
    fig, axes = plt.subplots(2, 1, figsize=(12,5), sharex=True)
    axes[0].plot(t, results[name], linewidth=0.8)
    axes[0].set_ylabel(name)
    axes[1].plot(t, results[name+'_filtered'], linewidth=0.8)
    axes[1].set_ylabel('Bandpass residual')
    axes[1].set_xlabel('Time (s)')
    fig.tight_layout()
    plt.show()

# Synthetic check: recover a periodic component without bridging a gap.
test_t = np.arange(2000)/200
v = np.sin(2*np.pi*3*test_t) + 0.5*np.sin(2*np.pi*0.2*test_t)
v[900:950] = np.nan
test_sos = signal.butter(3, [0.8,6], btype='bandpass', fs=200, output='sos')
f = filter_segments(v, test_sos, 400)
assert np.isnan(f[900:950]).all()
assert np.corrcoef(f[100:800], np.sin(2*np.pi*3*test_t[100:800]))[0,1] > 0.95

Plot the raw and filtered traces before calculating BPM. Motion magnitude can peak twice in one heartbeat. Check filter edge effects and the selected frequency range.

## 7. Heart-rate estimation

Use non-overlapping 10 s windows of uninterrupted intensity data. Motion magnitudes provide supporting evidence only, since two motion maxima per cycle are possible. FFT and Welch estimates are not independent validation. No interpolation across rejected frames.

In [ ]:
from scipy import signal
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
WINDOW_S = 10
LOW_HZ, HIGH_HZ = (0.8, 6.0)
MIN_SNR = 5.0
AGREEMENT_BPM = 15.0

def spectrum_rate(x, fs, method='welch'):
    if not np.isfinite(x).all() or np.std(x) < 1e-08:
        return (np.nan, np.nan)
    if method == 'welch':
        f, power = signal.welch(x, fs=fs, nperseg=min(len(x), round(5 * fs)))
    else:
        f = np.fft.rfftfreq(len(x), 1 / fs)
        power = abs(np.fft.rfft(signal.detrend(x) * np.hanning(len(x)))) ** 2
    band = (f >= LOW_HZ) & (f <= HIGH_HZ)
    if not band.any():
        return (np.nan, np.nan)
    fb, pb = (f[band], power[band])
    k = np.argmax(pb)
    return (60 * fb[k], pb[k] / max(np.median(pb), np.finfo(float).tiny))

def estimate_window(x, fs):
    out = dict(welch_bpm=np.nan, fft_bpm=np.nan, peak_bpm=np.nan, spectral_ratio=np.nan, bpm=np.nan, status='missing_signal')
    if not np.isfinite(x).all() or np.std(x) < 1e-08:
        return out
    w, ratio = spectrum_rate(x, fs)
    f, _ = spectrum_rate(x, fs, 'fft')
    peaks, _ = signal.find_peaks(x, distance=max(1, int(fs / HIGH_HZ)), prominence=0.5 * np.std(x))
    intervals = np.diff(peaks) / fs
    b = 60 / np.median(intervals) if len(intervals) >= 3 else np.nan
    out.update(welch_bpm=w, fft_bpm=f, peak_bpm=b, spectral_ratio=ratio)
    rates = np.array([w, f, b])
    if not np.isfinite(rates).all():
        out['status'] = 'insufficient_beats'
    elif ratio < MIN_SNR:
        out['status'] = 'weak_spectrum'
    elif np.ptp(rates) > AGREEMENT_BPM:
        out['status'] = 'method_disagreement'
    elif not LOW_HZ * 60 <= b <= HIGH_HZ * 60:
        out['status'] = 'outside_band'
    else:
        out.update(bpm=b, status='candidate')
    return out

def analyse_windows(table):
    t = table.time_s.to_numpy(float)
    fs = 1 / np.median(np.diff(t))
    if not np.allclose(np.diff(t), 1 / fs, rtol=1e-05, atol=1e-09) or fs <= 2 * HIGH_HZ:
        raise ValueError('Invalid time axis or sampling frequency')
    size = round(WINDOW_S * fs)
    rows = []
    for a in range(0, len(t), size):
        b = min(a + size, len(t))
        section = table.iloc[a:b]
        row = dict(start_frame=a, stop_frame=b, start_s=t[a], stop_s=t[b - 1] + 1 / fs)
        valid = section.valid.astype(str).str.lower().eq('true').to_numpy()
        row['valid_fraction'] = valid.mean()
        x = section.intensity_filtered.to_numpy(float)
        row.update(estimate_window(x, fs) if b - a == size and valid.all() else estimate_window(np.array([np.nan]), fs))
        if b - a < size:
            row['status'] = 'short_window'
        for name in ('frame_difference', 'flow_px_s'):
            rate, _ = spectrum_rate(section[name + '_filtered'].to_numpy(float), fs) if b - a == size and valid.all() else (np.nan, np.nan)
            row[name + '_bpm'] = rate
            primary = row['bpm']
            row[name + '_agreement'] = 'unavailable' if not np.isfinite(rate * primary) else 'fundamental' if abs(rate - primary) <= AGREEMENT_BPM else 'second_harmonic' if abs(rate - 2 * primary) <= AGREEMENT_BPM else 'disagreement'
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
# Synthetic checks, not biological validation.
fs_test=200
xt=np.sin(2*np.pi*3*np.arange(2000)/fs_test)
assert abs(estimate_window(xt,fs_test)['bpm']-180)<2
xt[100]=np.nan
assert np.isnan(estimate_window(xt,fs_test)['bpm'])
assert np.isnan(estimate_window(np.zeros(2000),fs_test)['bpm'])


## 8. Quality control and reporting

Candidate estimates are not calibrated probabilities. Reject windows with contradictory motion evidence; missing motion evidence is recorded separately. Thresholds are provisional. Retain every rejected window in the export.

In [ ]:
def quality_control(windows):
    windows = windows.copy()
    conflict = windows.frame_difference_agreement.eq('disagreement') & windows.flow_px_s_agreement.eq('disagreement')
    windows.loc[conflict & windows.status.eq('candidate'), 'status'] = 'motion_disagreement'
    windows['accepted'] = windows.status.eq('candidate')
    windows['bpm'] = windows.bpm.where(windows.accepted)
    return windows

def report_video(signal_file):
    signal_file = Path(signal_file)
    table = pd.read_csv(signal_file)
    if len(table) < 2:
        raise ValueError('Too few frames')
    windows = quality_control(analyse_windows(table))
    folder = signal_file.parent.parent / 'heart_rate'
    folder.mkdir(exist_ok=True)
    windows.to_csv(folder / 'window_estimates.csv', index=False)
    accepted = windows.loc[windows.accepted, 'bpm']
    summary = dict(video=folder.parent.name, windows=len(windows), accepted=len(accepted), coverage_s=float((windows.stop_s - windows.start_s).where(windows.accepted, 0).sum()), duration_s=float(windows.stop_s.iloc[-1] - windows.start_s.iloc[0]), median_bpm=accepted.median(), mean_bpm=accepted.mean())
    pd.DataFrame([summary]).to_csv(folder / 'summary.csv', index=False)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot((windows.start_s + windows.stop_s) / 2, windows.bpm, 'o-')
    for r in windows.loc[~windows.accepted].itertuples():
        ax.axvspan(r.start_s, r.stop_s, color='grey', alpha=0.2)
    ax.set(xlabel='Time (s)', ylabel='Candidate heart rate (BPM)')
    fig.tight_layout()
    fig.savefig(folder / 'heart_rate.png', dpi=150)
    plt.close(fig)
    return (summary, windows)
summary, windows = report_video(SIGNALS / 'cardiac_signals.csv')
print(summary)
print(windows.status.value_counts())


## 9. Validation

Annotate beat times independently on held-out recordings. Use the same start/stop boundaries as the exported windows. Reference BPM should be calculated from manually identified complete interbeat intervals (60 / mean interval). Record video, start_frame, stop_frame and reference_bpm in reference_rates.csv. Report rejection coverage as well as error. For repeated windows from the same larva, descriptive Bland–Altman limits are not independent-subject inferential limits.

In [ ]:
def validate_rates(predictions, reference, folder):
    keys = ['video', 'start_frame', 'stop_frame']
    if reference.duplicated(keys).any():
        raise ValueError('Duplicate reference windows')
    if not np.isfinite(reference.reference_bpm).all() or (reference.reference_bpm <= 0).any():
        raise ValueError('Invalid manual rates')
    paired = reference.merge(predictions[keys + ['bpm']], on=keys, how='left', validate='one_to_one')
    matched = paired.dropna(subset=['bpm']).copy()
    error = matched.bpm - matched.reference_bpm
    metrics = dict(reference_windows=len(paired), matched=len(matched), coverage=len(matched) / len(paired) if len(paired) else np.nan, mae=error.abs().mean(), rmse=np.sqrt((error ** 2).mean()), bias=error.mean())
    sd = error.std(ddof=1)
    metrics.update(loa_lower=error.mean() - 1.96 * sd, loa_upper=error.mean() + 1.96 * sd)
    paired.to_csv(folder / 'validation_pairs.csv', index=False)
    pd.DataFrame([metrics]).to_csv(folder / 'validation_metrics.csv', index=False)
    if len(matched) >= 2:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter((matched.bpm + matched.reference_bpm) / 2, error)
        for value in (metrics['bias'], metrics['loa_lower'], metrics['loa_upper']):
            ax.axhline(value, linestyle='--')
        ax.set(xlabel='Mean of methods (BPM)', ylabel='Estimated minus reference (BPM)')
        fig.tight_layout()
        fig.savefig(folder / 'bland_altman.png', dpi=150)
        plt.close(fig)
    return metrics


## 10. Batch analysis and reproducibility

Stages 1–6 must first produce matched registration and signal files for each recording. The batch runner below processes these saved signals, not raw videos. Detectron2 and DeepLabCut training remain separate runtime steps where dependencies require it. Use unique recording names.

In [ ]:
import json, platform, scipy, cv2
batch_folder = ROOT / 'outputs'
batch_folder.mkdir(exist_ok=True)
summaries = []
predictions = []
failures = []
for path in sorted(batch_folder.glob('*/signals/cardiac_signals.csv')):
    try:
        item, table = report_video(path)
        summaries.append(item)
        predictions.append(table.assign(video=item['video']))
    except Exception as exc:
        failures.append(dict(file=str(path), error=str(exc)))
pd.DataFrame(summaries).to_csv(batch_folder / 'batch_summary.csv', index=False)
pd.DataFrame(failures, columns=['file', 'error']).to_csv(batch_folder / 'batch_errors.csv', index=False)
manifest = dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__, opencv=cv2.__version__, pandas=pd.__version__, window_s=WINDOW_S, frequency_band_hz=[LOW_HZ, HIGH_HZ], min_spectral_ratio=MIN_SNR, agreement_bpm=AGREEMENT_BPM)
(batch_folder / 'analysis_parameters.json').write_text(json.dumps(manifest, indent=2))
reference_path = ROOT / 'reference_rates.csv'
if predictions:
    all_windows = pd.concat(predictions, ignore_index=True)
    all_windows.to_csv(batch_folder / 'all_windows.csv', index=False)
    if reference_path.exists():
        print(validate_rates(all_windows, pd.read_csv(reference_path), batch_folder))
    else:
        print('Validation pending: reference_rates.csv not supplied')
else:
    print('No successfully processed signal files')


### Checks before using the results

Check the acquisition FPS, model files, recording split and rejected windows. Agreement between spectra does not rule out a shared harmonic. Compare with manual beat counts.

## 11. Processing a recording

Use the trained fish model and matching full-frame DLC output. Detectron2 takes the original frames rather than the background-subtracted frames. Use a new output folder for each run.

In [ ]:
from pathlib import Path
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
ROI_SIZE = 96
MIN_LIKELIHOOD = .90
MIN_AXIS_PX = 10
RADIUS = 20
MIN_SEGMENT_S = 5
def register_heart(frame, heart, head, tail):
    landmarks = np.stack([heart, head, tail])
    if not np.isfinite(landmarks).all() or landmarks[:, 2].min() < MIN_LIKELIHOOD:
        return None, None, 'low_confidence'
    h, w = frame.shape[:2]
    if np.any(landmarks[:, :2] < 0) or np.any(landmarks[:, 0] >= w) or np.any(landmarks[:, 1] >= h):
        return None, None, 'outside_frame'
    axis = head[:2] - tail[:2]
    if np.linalg.norm(axis) < MIN_AXIS_PX:
        return None, None, 'short_body_axis'
    angle = np.degrees(np.arctan2(axis[1], axis[0]))
    centre = (ROI_SIZE - 1) / 2
    matrix = cv2.getRotationMatrix2D(tuple(heart[:2]), angle, 1.0)
    matrix[:, 2] += centre - heart[:2]
    size = (ROI_SIZE, ROI_SIZE)
    coverage = cv2.warpAffine(np.ones((h, w), np.uint8), matrix, size,
                              flags=cv2.INTER_NEAREST)
    if not coverage.all():
        return None, matrix, 'clipped_roi'
    roi = cv2.warpAffine(frame, matrix, size, flags=cv2.INTER_LINEAR)
    return roi, matrix, 'ok'



def extract_signals(images, accepted, region, fps):
    values = np.full((len(images), 3), np.nan)
    for i in range(len(images)):
        if not accepted[i]:
            continue
        current = np.asarray(images[i])
        values[i, 0] = current[region].mean()
        if i == 0 or not accepted[i-1]:
            continue
        previous = np.asarray(images[i-1])
        values[i, 1] = cv2.absdiff(previous, current)[region].mean()
        flow = cv2.calcOpticalFlowFarneback(previous, current, None,
                                             0.5, 3, 15, 3, 5, 1.2, 0)
        values[i, 2] = np.linalg.norm(flow[region], axis=1).mean() * fps
    return values


def filter_segments(values, sos, min_samples):
    result = np.full_like(values, np.nan, dtype=float)
    edges = np.flatnonzero(np.diff(np.r_[False, np.isfinite(values), False]))
    for start, stop in edges.reshape(-1, 2):
        if stop-start < min_samples:
            continue
        try:
            result[start:stop] = signal.sosfiltfilt(sos, values[start:stop])
        except ValueError:
            continue  # segment shorter than filter padding
    return result


from scipy import signal
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

WINDOW_S = 10
LOW_HZ, HIGH_HZ = 0.8, 6.0
MIN_SNR = 5.0
AGREEMENT_BPM = 15.0

def spectrum_rate(x, fs, method='welch'):
    if not np.isfinite(x).all() or np.std(x) < 1e-8:
        return np.nan, np.nan
    if method == 'welch':
        f, power = signal.welch(x, fs=fs, nperseg=min(len(x), round(5*fs)))
    else:
        f = np.fft.rfftfreq(len(x), 1/fs)
        power = abs(np.fft.rfft(signal.detrend(x)*np.hanning(len(x))))**2
    band = (f >= LOW_HZ) & (f <= HIGH_HZ)
    if not band.any():
        return np.nan, np.nan
    fb, pb = f[band], power[band]
    k = np.argmax(pb)
    return 60*fb[k], pb[k]/max(np.median(pb), np.finfo(float).tiny)

def estimate_window(x, fs):
    out = dict(welch_bpm=np.nan, fft_bpm=np.nan, peak_bpm=np.nan,
               spectral_ratio=np.nan, bpm=np.nan, status='missing_signal')
    if not np.isfinite(x).all() or np.std(x) < 1e-8:
        return out
    w, ratio = spectrum_rate(x, fs)
    f, _ = spectrum_rate(x, fs, 'fft')
    peaks, _ = signal.find_peaks(x, distance=max(1,int(fs/HIGH_HZ)), prominence=0.5*np.std(x))
    intervals = np.diff(peaks)/fs
    b = 60/np.median(intervals) if len(intervals)>=3 else np.nan
    out.update(welch_bpm=w, fft_bpm=f, peak_bpm=b, spectral_ratio=ratio)
    rates = np.array([w,f,b])
    if not np.isfinite(rates).all():
        out['status']='insufficient_beats'
    elif ratio < MIN_SNR:
        out['status']='weak_spectrum'
    elif np.ptp(rates)>AGREEMENT_BPM:
        out['status']='method_disagreement'
    elif not LOW_HZ*60 <= b <= HIGH_HZ*60:
        out['status']='outside_band'
    else:
        out.update(bpm=b, status='candidate')
    return out

def analyse_windows(table):
    t = table.time_s.to_numpy(float)
    fs = 1/np.median(np.diff(t))
    if not np.allclose(np.diff(t),1/fs,rtol=1e-5,atol=1e-9) or fs<=2*HIGH_HZ:
        raise ValueError('Invalid time axis or sampling frequency')
    size = round(WINDOW_S*fs)
    rows=[]
    for a in range(0,len(t),size):
        b=min(a+size,len(t)); section=table.iloc[a:b]
        row=dict(start_frame=a, stop_frame=b, start_s=t[a], stop_s=t[b-1]+1/fs)
        valid=section.valid.astype(str).str.lower().eq('true').to_numpy()
        row['valid_fraction']=valid.mean()
        x=section.intensity_filtered.to_numpy(float)
        row.update(estimate_window(x,fs) if b-a==size and valid.all() else estimate_window(np.array([np.nan]),fs))
        if b-a<size: row['status']='short_window'
        for name in ('frame_difference','flow_px_s'):
            rate,_=spectrum_rate(section[name+'_filtered'].to_numpy(float),fs) if b-a==size and valid.all() else (np.nan,np.nan)
            row[name+'_bpm']=rate
            primary=row['bpm']
            row[name+'_agreement']=('unavailable' if not np.isfinite(rate*primary) else
                'fundamental' if abs(rate-primary)<=AGREEMENT_BPM else
                'second_harmonic' if abs(rate-2*primary)<=AGREEMENT_BPM else 'disagreement')
        rows.append(row)
    return pd.DataFrame(rows)


def quality_control(windows):
    windows=windows.copy()
    conflict=(windows.frame_difference_agreement.eq('disagreement') &
              windows.flow_px_s_agreement.eq('disagreement'))
    windows.loc[conflict & windows.status.eq('candidate'),'status']='motion_disagreement'
    windows['accepted']=windows.status.eq('candidate')
    windows['bpm']=windows.bpm.where(windows.accepted)
    return windows

def report_video(signal_file):
    signal_file=Path(signal_file)
    table=pd.read_csv(signal_file)
    if len(table)<2: raise ValueError('Too few frames')
    windows=quality_control(analyse_windows(table))
    folder=signal_file.parent.parent/'heart_rate'
    folder.mkdir(exist_ok=True)
    windows.to_csv(folder/'window_estimates.csv',index=False)
    accepted=windows.loc[windows.accepted,'bpm']
    summary=dict(video=folder.parent.name, windows=len(windows), accepted=len(accepted),
                 coverage_s=float((windows.stop_s-windows.start_s).where(windows.accepted,0).sum()),
                 duration_s=float(windows.stop_s.iloc[-1]-windows.start_s.iloc[0]),
                 median_bpm=accepted.median(), mean_bpm=accepted.mean())
    pd.DataFrame([summary]).to_csv(folder/'summary.csv',index=False)
    fig,ax=plt.subplots(figsize=(10,3))
    ax.plot((windows.start_s+windows.stop_s)/2,windows.bpm,'o-')
    for r in windows.loc[~windows.accepted].itertuples():
        ax.axvspan(r.start_s,r.stop_s,color='grey',alpha=.2)
    ax.set(xlabel='Time (s)',ylabel='Candidate heart rate (BPM)')
    fig.tight_layout(); fig.savefig(folder/'heart_rate.png',dpi=150); plt.close(fig)
    return summary,windows



def validate_rates(predictions, reference, folder):
    keys=['video','start_frame','stop_frame']
    if reference.duplicated(keys).any(): raise ValueError('Duplicate reference windows')
    if not np.isfinite(reference.reference_bpm).all() or (reference.reference_bpm<=0).any():
        raise ValueError('Invalid manual rates')
    paired=reference.merge(predictions[keys+['bpm']],on=keys,how='left',validate='one_to_one')
    matched=paired.dropna(subset=['bpm']).copy()
    error=matched.bpm-matched.reference_bpm
    metrics=dict(reference_windows=len(paired),matched=len(matched),
                 coverage=len(matched)/len(paired) if len(paired) else np.nan,
                 mae=error.abs().mean(),rmse=np.sqrt((error**2).mean()),bias=error.mean())
    sd=error.std(ddof=1)
    metrics.update(loa_lower=error.mean()-1.96*sd,loa_upper=error.mean()+1.96*sd)
    paired.to_csv(folder/'validation_pairs.csv',index=False)
    pd.DataFrame([metrics]).to_csv(folder/'validation_metrics.csv',index=False)
    if len(matched)>=2:
        fig,ax=plt.subplots(figsize=(7,5))
        ax.scatter((matched.bpm+matched.reference_bpm)/2,error)
        for value in (metrics['bias'],metrics['loa_lower'],metrics['loa_upper']): ax.axhline(value,linestyle='--')
        ax.set(xlabel='Mean of methods (BPM)',ylabel='Estimated minus reference (BPM)')
        fig.tight_layout();fig.savefig(folder/'bland_altman.png',dpi=150);plt.close(fig)
    return metrics


In [ ]:
def load_pose(path):
    pose = pd.read_hdf(path) if path.suffix == '.h5' else pd.read_csv(path, header=[0, 1, 2], index_col=0)
    if pose.columns.nlevels != 3 or len(pose.columns.get_level_values(0).unique()) != 1:
        raise ValueError('Expected single-animal, single-scorer pose table')
    if not np.array_equal(pose.index.to_numpy(), np.arange(len(pose))):
        raise ValueError('Pose frame indices do not match original video')
    pose = pose.droplevel(0, axis=1)
    return [pose[p][['x', 'y', 'likelihood']].to_numpy(float) for p in ('heart_center', 'head', 'tail_base')]

def process_recording(video, pose_file, predictor, fish_class, output, fps):
    video, pose_file, output = map(Path, (video, pose_file, output))
    if not np.isfinite(fps) or fps <= 2 * HIGH_HZ:
        raise ValueError('Supply verified acquisition FPS')
    points = load_pose(pose_file)
    count = len(points[0])
    if count < 2:
        raise ValueError('Insufficient frames')
    output.mkdir(parents=True, exist_ok=False)
    reg = output / 'registration'
    reg.mkdir()
    sig = output / 'signals'
    sig.mkdir()
    cap = cv2.VideoCapture(str(video))
    rows = []
    rois = np.lib.format.open_memmap(reg / 'cardiac_rois.npy', mode='w+', dtype=np.uint8, shape=(count, ROI_SIZE, ROI_SIZE))
    try:
        for i in range(count):
            ok, frame = cap.read()
            if not ok:
                raise ValueError(f'Video ended at frame {i}')
            heart, head, tail = (p[i] for p in points)
            inst = predictor(frame)['instances'].to('cpu')
            fish = inst[inst.pred_classes == fish_class]
            roi, matrix, status = register_heart(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), heart, head, tail)
            score = np.nan
            if len(fish) != 1:
                status = 'fish_missing_or_multiple'
            else:
                score = float(fish.scores[0])
                if status == 'ok':
                    x, y = np.floor(heart[:2]).astype(int)
                    if not bool(fish.pred_masks[0, y, x]):
                        status = 'heart_outside_fish_mask'
            valid = status == 'ok'
            rois[i] = roi if valid else 0
            row = dict(frame=i, time_s=i / fps, valid=valid, status=status, fish_score=score, heart_x=heart[0], heart_y=heart[1], heart_likelihood=heart[2])
            row.update({f'm{j}{k}': matrix[j, k] if matrix is not None else np.nan for j in range(2) for k in range(3)})
            rows.append(row)
        if cap.read()[0]:
            raise ValueError('Video longer than pose table')
    finally:
        cap.release()
        rois.flush()
    qc = pd.DataFrame(rows)
    qc.to_csv(reg / 'registration_qc.csv', index=False)
    y, x = np.ogrid[:ROI_SIZE, :ROI_SIZE]
    region = (x - (ROI_SIZE - 1) / 2) ** 2 + (y - (ROI_SIZE - 1) / 2) ** 2 <= RADIUS ** 2
    raw = extract_signals(rois, qc.valid.to_numpy(), region, fps)
    sos = signal.butter(3, [LOW_HZ, HIGH_HZ], btype='bandpass', fs=fps, output='sos')
    results = qc[['frame', 'time_s', 'valid']].copy()
    for j, name in enumerate(['intensity', 'frame_difference', 'flow_px_s']):
        results[name] = raw[:, j]
        results[name + '_filtered'] = filter_segments(raw[:, j], sos, int(np.ceil(MIN_SEGMENT_S * fps)))
    results.to_csv(sig / 'cardiac_signals.csv', index=False)
    summary, windows = report_video(sig / 'cardiac_signals.csv')
    (output / 'run.json').write_text(json.dumps(dict(video=str(video), pose=str(pose_file), fps=fps, fish_class=fish_class, roi_size=ROI_SIZE, likelihood=MIN_LIKELIHOOD, band=[LOW_HZ, HIGH_HZ], window_s=WINDOW_S), indent=2))
    return (summary, windows)


### Pose inference (DeepLabCut 2.x runtime)
Run in the compatible DLC environment before running the batch below. Review the generated pose files and enter their exact paths in the recording manifest.

In [ ]:
RUN_POSE = False
if RUN_POSE:
    import deeplabcut
    config = '/content/drive/MyDrive/ZebrafishHeartRate/deeplabcut/config.yaml'
    videos = ['/content/drive/MyDrive/ZebrafishHeartRate/videos/recording.avi']
    deeplabcut.analyze_videos(config, videos, save_as_csv=True)


In [ ]:
# Detectron2 inference runtime
RUN_BATCH = False
if RUN_BATCH:
    from google.colab import drive
    from detectron2.config import get_cfg
    from detectron2.engine import DefaultPredictor
    drive.mount('/content/drive')
    root = Path('/content/drive/MyDrive/ZebrafishHeartRate')
    model = root/'detectron2'/'Model'
    cfg = get_cfg(); cfg.merge_from_file(str(model/'cfg.yaml'))
    cfg.MODEL.WEIGHTS = str(model/'model_final.pth')
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = .5
    predictor = DefaultPredictor(cfg)
    # CSV columns: recording, video, pose, fps, fish_class (verify class index from training metadata)
    manifest = pd.read_csv(root/'recordings.csv')
    if manifest.recording.duplicated().any():
        raise ValueError('Duplicate recording names')
    from datetime import datetime, timezone
    run = root/'runs'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run.mkdir(parents=True)
    summaries, tables, errors = [], [], []
    for r in manifest.itertuples(index=False):
        try:
            if Path(str(r.recording)).name != str(r.recording) or str(r.recording) in ('.','..'):
                raise ValueError('Invalid recording name')
            summary, windows = process_recording(root/r.video,root/r.pose,predictor,int(r.fish_class),run/str(r.recording),float(r.fps))
            summaries.append(summary); tables.append(windows.assign(video=r.recording))
        except Exception as exc:
            errors.append(dict(recording=r.recording,error=str(exc)))
    pd.DataFrame(summaries).to_csv(run/'batch_summary.csv',index=False)
    pd.DataFrame(errors,columns=['recording','error']).to_csv(run/'batch_errors.csv',index=False)
    if tables:
        all_windows = pd.concat(tables,ignore_index=True)
        all_windows.to_csv(run/'all_windows.csv',index=False)
        if (root/'reference_rates.csv').exists():
            print(validate_rates(all_windows,pd.read_csv(root/'reference_rates.csv'),run))


## 12. Residual motion audit

Use an independently reviewed binary mask of stable, non-cardiac tissue in the registered ROI. Save it as a PNG (white = reference tissue). Background alone is not a suitable reference. Translation estimates are diagnostic; no additional cardiac warping is applied. Thresholds are provisional and do not establish biological accuracy.

In [ ]:
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

# Reference mask must be drawn on stable non-cardiac tissue in registered coordinates.
MAX_SHIFT_PX = 2.0
MIN_CORRELATION = 0.7
MIN_TEXTURE_SD = 3.0


def residual_motion(images, valid, reference_mask):
    mask = np.asarray(reference_mask, dtype=bool)
    if mask.shape != images.shape[1:] or mask.sum() < 100:
        raise ValueError('Reference mask must match ROI and contain at least 100 pixels')
    # Exclude the central cardiac analysis region.
    y, x = np.ogrid[:mask.shape[0], :mask.shape[1]]
    if np.any(mask & ((x-(mask.shape[1]-1)/2)**2 + (y-(mask.shape[0]-1)/2)**2 <= 20**2)):
        raise ValueError('Reference mask overlaps cardiac region')
    rows = []
    for i in range(len(images)):
        row = dict(frame=i, dx_px=np.nan, dy_px=np.nan, shift_px=np.nan,
                   correlation=np.nan, motion_status='missing_pair')
        if i and valid[i] and valid[i-1]:
            a, b = (np.asarray(images[j], dtype=np.float32) for j in (i-1,i))
            if min(a[mask].std(), b[mask].std()) < MIN_TEXTURE_SD:
                row['motion_status'] = 'low_texture'
            else:
                try:
                    cc, warp = cv2.findTransformECC(a, b, np.eye(2,3,dtype=np.float32),
                        cv2.MOTION_TRANSLATION, (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 80, 1e-5),
                        mask.astype(np.uint8)*255, 5)
                    dx, dy = map(float, warp[:,2]); shift = float(np.hypot(dx,dy))
                    status = 'ok' if cc >= MIN_CORRELATION and shift <= MAX_SHIFT_PX else 'residual_motion'
                    row.update(dx_px=dx,dy_px=dy,shift_px=shift,correlation=float(cc),motion_status=status)
                except cv2.error:
                    row['motion_status'] = 'registration_failed'
        rows.append(row)
    return pd.DataFrame(rows)


def audit_recording(folder, reference_mask):
    folder = Path(folder)
    images = np.load(folder/'registration/cardiac_rois.npy', mmap_mode='r')
    qc = pd.read_csv(folder/'registration/registration_qc.csv')
    if len(qc) != len(images) or not np.array_equal(qc.frame, np.arange(len(images))):
        raise ValueError('Registration files do not align')
    valid = qc.valid.astype(str).str.lower().eq('true').to_numpy()
    motion = residual_motion(images, valid, reference_mask)
    motion['time_s'] = qc.time_s
    out = folder/'motion_audit'; out.mkdir(exist_ok=True)
    motion.to_csv(out/'residual_motion.csv',index=False)
    windows = pd.read_csv(folder/'heart_rate/window_estimates.csv')
    # Keep original estimates; report conservative additional screening separately.
    windows['motion_valid_fraction'] = [motion.motion_status.iloc[int(a)+1:int(b)].eq('ok').mean()
        for a,b in zip(windows.start_frame,windows.stop_frame)]
    windows['motion_screened_bpm'] = windows.bpm.where(windows.motion_valid_fraction >= .95)
    windows.to_csv(out/'screened_windows.csv',index=False)
    fig, ax = plt.subplots(figsize=(10,3))
    ax.plot(motion.time_s,motion.shift_px,linewidth=.7)
    ax.axhline(MAX_SHIFT_PX,linestyle='--'); ax.set(xlabel='Time (s)',ylabel='Residual shift (px/frame)')
    fig.tight_layout(); fig.savefig(out/'residual_motion.png',dpi=150); plt.close(fig)
    return motion, windows


In [ ]:
# Synthetic translation check
rng = np.random.default_rng(12)
a = cv2.GaussianBlur(rng.integers(0,256,(96,96),dtype=np.uint8),(5,5),0)
b_test = cv2.warpAffine(a,np.float32([[1,0,1],[0,1,0]]),(96,96))
mask = np.zeros((96,96),bool); mask[8:24,10:80] = True
check = residual_motion(np.stack([a,b_test]),[True,True],mask)
assert abs(check.shift_px.iloc[1]-1) < .4
assert check.motion_status.iloc[1] == 'ok'
assert residual_motion(np.stack([a,b_test]),[True,False],mask).motion_status.iloc[1] == 'missing_pair'
print(check)

In [ ]:
RUN_MOTION_AUDIT = False
if RUN_MOTION_AUDIT:
    folder = Path('/content/drive/MyDrive/ZebrafishHeartRate/runs/REPLACE_RUN/REPLACE_RECORDING')
    reference = cv2.imread(str(folder/'reference_mask.png'),cv2.IMREAD_GRAYSCALE)
    if reference is None:
        raise FileNotFoundError('Supply reviewed reference_mask.png')
    motion, screened = audit_recording(folder, reference > 127)
    display(screened)

## 13. Adaptive cardiac region

Estimate cardiac radius from the median head–tail-base distance within each analysis window. Keep the radius fixed within that window; reject frames with insufficient fish-mask coverage. This is a scale proxy, not a segmentation of the heart. Compare with the fixed-radius baseline on held-out recordings. Run the Stage 11 definitions first. The new runner writes to a separate output directory.

In [ ]:
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from scipy import signal

# Use a constant mask within each analysis window to avoid artificial
# intensity changes from a changing pixel population.
AXIS_FRACTION = 0.18
MIN_RADIUS, MAX_RADIUS = 8, 32
MIN_MASK_FRACTION = 0.80


def adaptive_region(axes, size=96):
    axes = np.asarray(axes, dtype=float)
    if len(axes) == 0 or not np.isfinite(axes).all() or np.min(axes) <= 0:
        raise ValueError('Invalid anatomical scale')
    radius = int(np.clip(round(np.median(axes)*AXIS_FRACTION), MIN_RADIUS, MAX_RADIUS))
    y, x = np.ogrid[:size, :size]
    centre = (size-1)/2
    return (x-centre)**2+(y-centre)**2 <= radius**2, radius


def adaptive_signals(rois, masks, qc, axes, fps):
    if len(rois) != len(masks) or len(rois) != len(qc) or len(rois) != len(axes):
        raise ValueError('Frame counts differ')
    if not np.isfinite(fps) or fps <= 2*HIGH_HZ:
        raise ValueError('Invalid FPS')
    values = np.full((len(rois), 3), np.nan)
    valid = qc.valid.to_numpy(dtype=bool).copy()
    audit = []
    step = round(WINDOW_S*fps)
    for a in range(0, len(rois), step):
        b = min(a+step,len(rois))
        usable = valid[a:b] & np.isfinite(axes[a:b]) & (axes[a:b]>0)
        if not usable.any():
            valid[a:b] = False
            audit.append(dict(start_frame=a, stop_frame=b, radius_px=np.nan, valid_fraction=0))
            continue
        region, radius = adaptive_region(axes[a:b][usable], rois.shape[1])
        coverage = np.array([np.mean(m[region]>0) for m in masks[a:b]])
        accepted = usable & (coverage >= MIN_MASK_FRACTION)
        valid[a:b] = accepted
        raw = extract_signals(rois[a:b], accepted, region, fps)
        values[a:b] = raw
        audit.append(dict(start_frame=a,stop_frame=b,radius_px=radius,valid_fraction=accepted.mean()))
    table = qc[['frame','time_s']].copy()
    table['valid'] = valid
    sos = signal.butter(3,[LOW_HZ,HIGH_HZ],btype='bandpass',fs=fps,output='sos')
    for j,name in enumerate(['intensity','frame_difference','flow_px_s']):
        table[name] = values[:,j]
        table[name+'_filtered'] = np.nan
        for a in range(0,len(rois),step):
            b=min(a+step,len(rois))
            table.loc[table.index[a:b],name+'_filtered'] = filter_segments(values[a:b,j],sos,int(np.ceil(MIN_SEGMENT_S*fps)))
    return table,pd.DataFrame(audit)


In [ ]:
def process_adaptive_recording(video, pose_file, predictor, fish_class, output, fps):
    video, pose_file, output = map(Path, (video, pose_file, output))
    if not np.isfinite(fps) or fps <= 2 * HIGH_HZ:
        raise ValueError('Supply verified acquisition FPS')
    points = load_pose(pose_file)
    count = len(points[0])
    if count < 2:
        raise ValueError('Insufficient frames')
    output.mkdir(parents=True, exist_ok=False)
    reg = output / 'registration'
    reg.mkdir()
    sig = output / 'signals'
    sig.mkdir()
    cap = cv2.VideoCapture(str(video))
    rows = []
    axes = np.full(count, np.nan)
    masks = np.lib.format.open_memmap(reg / 'fish_masks.npy', mode='w+', dtype=np.uint8, shape=(count, ROI_SIZE, ROI_SIZE))
    rois = np.lib.format.open_memmap(reg / 'cardiac_rois.npy', mode='w+', dtype=np.uint8, shape=(count, ROI_SIZE, ROI_SIZE))
    try:
        for i in range(count):
            ok, frame = cap.read()
            if not ok:
                raise ValueError(f'Video ended at frame {i}')
            heart, head, tail = (p[i] for p in points)
            inst = predictor(frame)['instances'].to('cpu')
            fish = inst[inst.pred_classes == fish_class]
            roi, matrix, status = register_heart(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), heart, head, tail)
            score = np.nan
            if len(fish) != 1:
                status = 'fish_missing_or_multiple'
            else:
                score = float(fish.scores[0])
                if status == 'ok':
                    x, y = np.floor(heart[:2]).astype(int)
                    if not bool(fish.pred_masks[0, y, x]):
                        status = 'heart_outside_fish_mask'
            valid = status == 'ok'
            rois[i] = roi if valid else 0
            masks[i] = 0
            if valid:
                axes[i] = np.linalg.norm(head[:2] - tail[:2])
                masks[i] = cv2.warpAffine(fish.pred_masks[0].numpy().astype(np.uint8), matrix, (ROI_SIZE, ROI_SIZE), flags=cv2.INTER_NEAREST)
            row = dict(frame=i, time_s=i / fps, valid=valid, status=status, fish_score=score, heart_x=heart[0], heart_y=heart[1], heart_likelihood=heart[2])
            row.update({f'm{j}{k}': matrix[j, k] if matrix is not None else np.nan for j in range(2) for k in range(3)})
            rows.append(row)
        if cap.read()[0]:
            raise ValueError('Video longer than pose table')
    finally:
        cap.release()
        rois.flush()
        masks.flush()
    qc = pd.DataFrame(rows)
    qc.to_csv(reg / 'registration_qc.csv', index=False)
    results, audit = adaptive_signals(rois, masks, qc, axes, fps)
    audit.to_csv(reg / 'adaptive_roi_qc.csv', index=False)
    np.save(reg / 'body_axis_px.npy', axes)
    results.to_csv(sig / 'cardiac_signals.csv', index=False)
    summary, windows = report_video(sig / 'cardiac_signals.csv')
    (output / 'run.json').write_text(json.dumps(dict(video=str(video), pose=str(pose_file), fps=fps, fish_class=fish_class, roi_size=ROI_SIZE, likelihood=MIN_LIKELIHOOD, band=[LOW_HZ, HIGH_HZ], window_s=WINDOW_S, axis_fraction=AXIS_FRACTION, min_radius=MIN_RADIUS, max_radius=MAX_RADIUS, min_mask_fraction=MIN_MASK_FRACTION), indent=2))
    return (summary, windows)


In [ ]:
# Scale and mask geometry checks
small, r1 = adaptive_region([60, 61, 59])
large, r2 = adaptive_region([120, 121, 119])
assert r2 > r1 and large.sum() > small.sum()
assert adaptive_region([1000])[1] == MAX_RADIUS
print('Radius checks:', r1, r2)


Use `process_adaptive_recording(video, pose_file, predictor, fish_class, output, fps)` with the same inputs as Stage 11 and a new output directory. Review the adaptive masks and radius per window. Run Stage 12 separately with a reviewed reference mask; its screening is not automatically applied to these estimates. The scale fraction and coverage threshold require calibration.

## 14. Fundamental frequency
Compare signed intensity and motion spectra. Motion magnitude can peak twice per cycle. Reject competing fundamentals rather than automatically halving a peak.

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
from pathlib import Path
import json

def spectral_candidates(x, fs, low=0.8, high=6.0):
    x = np.asarray(x, float)
    if len(x) < fs * 5 or not np.isfinite(x).all() or np.std(x) < 1e-08:
        return (np.empty(0), np.empty(0))
    f, p = signal.welch(x, fs=fs, nperseg=min(len(x), round(5 * fs)))
    keep = (f >= low) & (f <= high)
    return (f[keep], p[keep] / max(np.median(p[keep]), np.finfo(float).tiny))

def harmonic_rate(intensity, motions, fs, min_ratio=5.0, ambiguity=0.65):
    f, p = spectral_candidates(intensity, fs)
    if not len(f):
        return (np.nan, 'missing_intensity')
    peaks, _ = signal.find_peaks(np.r_[0, p, 0])
    peaks = peaks - 1
    peaks = peaks[(peaks >= 0) & (peaks < len(p))]
    peaks = peaks[p[peaks] >= min_ratio]
    if not len(peaks):
        return (np.nan, 'weak_spectrum')
    order = peaks[np.argsort(p[peaks])[::-1]]
    k = order[0]
    tol = max(0.21, f[1] - f[0] if len(f) > 1 else 0.21)
    for j in order[1:]:
        if p[j] >= ambiguity * p[k]:
            return (np.nan, 'ambiguous_intensity')
        if f[j] < f[k] and abs(2 * f[j] - f[k]) <= tol and (p[j] >= 0.12 * p[k]):
            return (np.nan, 'possible_double_frequency')
    support = 0
    conflict = 0
    for x in motions:
        fm, pm = spectral_candidates(x, fs)
        if len(fm) and pm.max() >= min_ratio:
            q = fm[pm.argmax()]
            if min(abs(q - f[k]), abs(q - 2 * f[k])) <= tol:
                support += 1
            else:
                conflict += 1
    if not support or conflict:
        return (np.nan, 'motion_unconfirmed')
    return (60 * f[k], 'accepted')

def harmonic_windows(table, window_s=10, min_ratio=5.0, ambiguity=0.65):
    t = table.time_s.to_numpy(float)
    if len(t) < 2:
        raise ValueError('Too few frames')
    fs = 1 / np.median(np.diff(t))
    if fs <= 12 or not np.allclose(np.diff(t), 1 / fs):
        raise ValueError('Invalid sampling')
    if not np.array_equal(table.frame, np.arange(len(table))):
        raise ValueError('Frame indices must be contiguous')
    step = round(window_s * fs)
    rows = []
    for a in range(0, len(t), step):
        b = min(a + step, len(t))
        q = table.iloc[a:b]
        bpm, status = (np.nan, 'invalid_window')
        if b - a == step and q.valid.astype(str).str.lower().eq('true').all():
            bpm, status = harmonic_rate(q.intensity_filtered, [q.frame_difference_filtered, q.flow_px_s_filtered], fs, min_ratio, ambiguity)
        rows.append(dict(start_frame=a, stop_frame=b, start_s=t[a], stop_s=t[b - 1] + 1 / fs, bpm=bpm, status=status))
    return pd.DataFrame(rows)


## 15. Multiple larvae
Associate instance masks across frames with gated centroid assignment. Tracks are provisional tracklets, not guaranteed biological identities. End all affected tracklets during ambiguous crossings; never bridge a missing frame. Cardiac landmarks must be supplied per instance by a multi-animal pose model and reviewed before per-track signal extraction.

In [ ]:
from scipy.optimize import linear_sum_assignment

def associate_centroids(frames, max_distance=30., margin=5.):
    active={}; next_id=0; rows=[]
    for frame, centroids in enumerate(frames):
        xy=np.asarray(centroids,float).reshape(-1,2)
        if not np.isfinite(xy).all(): raise ValueError('Nonfinite centroid')
        ids=list(active); assigned={}; ambiguous=set()
        if ids and len(xy):
            d=np.linalg.norm(np.asarray([active[i] for i in ids])[:,None,:]-xy[None,:,:],axis=2)
            for i in range(len(ids)):
                order=np.argsort(d[i]); near=order[d[i,order]<=max_distance]
                if len(near)>1 and d[i,near[1]]-d[i,near[0]]<margin: ambiguous.update(near.tolist())
            for j in range(len(xy)):
                near=np.sort(d[:,j]); near=near[near<=max_distance]
                if len(near)>1 and near[1]-near[0]<margin: ambiguous.add(j)
            cost=np.where(d<=max_distance,d,1e9)
            # Block tracks touching an ambiguous detection.
            if ambiguous:
                blocked=(d[:,list(ambiguous)]<=max_distance).any(axis=1)
                cost[blocked,:]=1e9; cost[:,list(ambiguous)]=1e9
            for i,j in zip(*linear_sum_assignment(cost)):
                if cost[i,j]<1e9: assigned[j]=ids[i]
        active={}
        for j,point in enumerate(xy):
            if j in ambiguous:
                rows.append(dict(frame=frame,detection=j,track_id=-1,x=point[0],y=point[1],status='ambiguous')); continue
            if j not in assigned: assigned[j]=next_id; next_id+=1
            k=assigned[j]; active[k]=point
            rows.append(dict(frame=frame,detection=j,track_id=k,x=point[0],y=point[1],status='tracklet'))
    return pd.DataFrame(rows,columns=['frame','detection','track_id','x','y','status'])

def track_video(video,predictor,fish_class,max_distance=30.,margin=5.):
    import cv2
    cap=cv2.VideoCapture(str(video)); frames=[]
    if not cap.isOpened(): raise FileNotFoundError(video)
    try:
        while True:
            ok,frame=cap.read()
            if not ok: break
            inst=predictor(frame)['instances'].to('cpu')
            masks=inst[inst.pred_classes==fish_class].pred_masks.numpy()
            points=[]
            for mask in masks:
                y,x=np.nonzero(mask)
                if not len(x): raise ValueError('Empty instance mask')
                points.append([x.mean(),y.mean()])
            frames.append(points)
    finally: cap.release()
    return associate_centroids(frames,max_distance,margin)


## 16. Parameter selection
Tune spectral thresholds on development recordings only. The reference manifest uses video, animal_id, start_frame, stop_frame and reference_bpm. Split by animal (and recording), not by adjacent frames. Penalise missing estimates explicitly.

In [ ]:
from itertools import product
KEYS = ['video', 'start_frame', 'stop_frame']

def select_parameters(tables, reference, min_coverage=0.8, missing_penalty=60.0):
    if reference.empty or reference.duplicated(KEYS).any():
        raise ValueError('Invalid reference keys')
    if set(reference.video) != set(tables):
        raise ValueError('Development recordings differ')
    if not np.isfinite(reference.reference_bpm).all():
        raise ValueError('Invalid reference')
    rows = []
    for ratio, ambiguity in product([3.0, 5.0, 8.0], [0.5, 0.65, 0.8]):
        pred = pd.concat([harmonic_windows(t, min_ratio=ratio, ambiguity=ambiguity).assign(video=v) for v, t in tables.items()])
        m = reference.merge(pred[KEYS + ['bpm']], on=KEYS, how='left', validate='one_to_one')
        e = (m.bpm - m.reference_bpm).abs()
        coverage = e.notna().mean()
        loss = e.fillna(missing_penalty).mean()
        rows.append(dict(min_ratio=ratio, ambiguity=ambiguity, coverage=coverage, mae=e.mean(), loss=loss))
    results = pd.DataFrame(rows).sort_values('loss')
    eligible = results[results.coverage >= min_coverage]
    if eligible.empty:
        raise ValueError('No setting meets development coverage requirement')
    return (eligible.iloc[0].to_dict(), results)


## 17. Held-out validation
Report agreement on matched windows and coverage on all reference windows. Bootstrap whole animals to account for repeated windows. Limits of agreement below are descriptive pooled limits, not a repeated-measures model.

In [ ]:
def independent_validation(pred, ref, development_ref, folder, repeats=2000, seed=42):
    for r in (ref, development_ref):
        if r[['video', 'animal_id']].isna().any().any():
            raise ValueError('Missing grouping identifiers')
    for key in ('video', 'animal_id'):
        if set(ref[key]) & set(development_ref[key]):
            raise ValueError('Development/test overlap: ' + key)
    if ref.empty or ref.duplicated(KEYS).any() or pred.duplicated(KEYS).any():
        raise ValueError('Invalid validation keys')
    if not np.isfinite(ref.reference_bpm).all() or (ref.reference_bpm <= 0).any():
        raise ValueError('Invalid reference rates')
    pairs = ref.merge(pred[KEYS + ['bpm']], on=KEYS, how='left', validate='one_to_one')
    pairs['error'] = pairs.bpm - pairs.reference_bpm

    def metrics(q):
        e = q.error.dropna()
        sd = e.std(ddof=1)
        return dict(coverage=len(e) / len(q), mae=e.abs().mean(), rmse=np.sqrt((e ** 2).mean()), bias=e.mean(), loa_lower=e.mean() - 1.96 * sd, loa_upper=e.mean() + 1.96 * sd)
    result = metrics(pairs)
    groups = [g for _, g in pairs.groupby('animal_id')]
    rng = np.random.default_rng(seed)
    if len(groups) >= 2:
        draws = pd.DataFrame([metrics(pd.concat([groups[i] for i in rng.integers(0, len(groups), len(groups))])) for _ in range(repeats)])
        for k in result.copy():
            result[k + '_ci_low'] = draws[k].quantile(0.025)
            result[k + '_ci_high'] = draws[k].quantile(0.975)
    result.update(animals=len(groups), reference_windows=len(pairs), matched=pairs.error.notna().sum())
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    pairs.to_csv(folder / 'heldout_pairs.csv', index=False)
    pd.DataFrame([result]).to_csv(folder / 'heldout_metrics.csv', index=False)
    return result


## 18. Final analysis runner
This runner uses the existing single-larva adaptive recording function and precomputed DeepLabCut pose files. A reviewed reference mask is required for residual motion screening and must exclude the maximum adaptive cardiac radius. Multi-larva exports remain tracklet diagnostics only. Train/test separation, acquisition FPS and checkpoint provenance must be checked before execution.

In [ ]:
def final_recording(video, pose_file, predictor, fish_class, output, fps, reference_mask, parameters):
    output = Path(output)
    mask = np.asarray(reference_mask, bool)
    y, x = np.ogrid[:mask.shape[0], :mask.shape[1]]
    if mask.shape != (ROI_SIZE, ROI_SIZE) or np.any(mask & ((x - (ROI_SIZE - 1) / 2) ** 2 + (y - (ROI_SIZE - 1) / 2) ** 2 <= MAX_RADIUS ** 2)):
        raise ValueError('Reference mask overlaps adaptive ROI or has wrong shape')
    process_adaptive_recording(video, pose_file, predictor, fish_class, output, fps)
    table = pd.read_csv(output / 'signals/cardiac_signals.csv')
    windows = harmonic_windows(table, min_ratio=parameters['min_ratio'], ambiguity=parameters['ambiguity'])
    motion, _ = audit_recording(output, mask)
    windows['motion_valid_fraction'] = [motion.motion_status.iloc[a + 1:b].eq('ok').mean() for a, b in zip(windows.start_frame, windows.stop_frame)]
    reject = windows.motion_valid_fraction.lt(0.95) & windows.bpm.notna()
    windows.loc[reject, 'status'] = 'residual_motion'
    windows.loc[reject, 'bpm'] = np.nan
    windows['video'] = Path(video).stem
    windows.to_csv(output / 'final_windows.csv', index=False)
    (output / 'final_parameters.json').write_text(json.dumps(dict(parameters=parameters, fps=fps, minimum_motion_fraction=0.95), indent=2))
    return windows


In [ ]:
# Checks using simulated data
fs=200; t=np.arange(2000)/fs
x=np.sin(2*np.pi*3*t); m=np.sin(2*np.pi*6*t)
bpm,status=harmonic_rate(x,[m,m],fs)
assert abs(bpm-180)<1 and status=='accepted'
assert np.isnan(harmonic_rate(x,[np.sin(2*np.pi*2*t)],fs)[0])
assert np.isnan(harmonic_rate(np.ones_like(x),[m],fs)[0])
tracks=associate_centroids([[[0,0],[100,0]],[[2,0],[98,0]],[],[[3,0]]])
assert tracks.track_id.tolist()==[0,1,0,1,2]
print('Synthetic frequency and tracklet checks passed')


### Running the combined analysis

Keep the videos, annotations, split lists, pose files, checkpoints and camera metadata. Fix the analysis parameters before using the test set. Multi-fish tracking is separate from the single-fish BPM calculation.

## 19. Experimental deployment
Use one row per recording in a manifest. Paths are absolute or relative to the manifest directory. Acquisition FPS must come from camera metadata, not playback speed. Use a manually reviewed mask for each recording. Keep held-out animals separate from model training and parameter selection.

In [ ]:
import hashlib

def file_digest(path):
    with open(path, 'rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def audit_manifest(path):
    path = Path(path)
    data = pd.read_csv(path, dtype={'animal_id': str, 'video': str})
    required = ['video', 'animal_id', 'pose_file', 'reference_mask', 'fps', 'split']
    if data[required].isna().any().any() or data.empty:
        raise ValueError('Incomplete manifest')
    if not data.split.isin(['development', 'test']).all():
        raise ValueError('Unknown split')
    if data.groupby('animal_id').split.nunique().gt(1).any():
        raise ValueError('Animal split overlap')
    if not np.isfinite(data.fps).all() or (data.fps <= 12).any():
        raise ValueError('Invalid acquisition FPS for 6 Hz analysis')
    for col in ['video', 'pose_file', 'reference_mask']:
        data[col] = data[col].map(lambda p: str((path.parent / p).resolve()))
    if data.video.map(lambda p: Path(p).stem).duplicated().any():
        raise ValueError('Recording stems must be unique')
    return data

def deploy_recordings(manifest, predictor, fish_class, output, parameters, checkpoint, split='development'):
    data = audit_manifest(manifest)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=False)
    checkpoint_hash = file_digest(checkpoint)
    (output / 'run_config.json').write_text(json.dumps({'checkpoint': str(checkpoint), 'checkpoint_sha256': checkpoint_hash, 'parameters': parameters, 'split': split}, indent=2))
    data.to_csv(output / 'manifest.csv', index=False)
    results, logs = ([], [])
    for r in data.loc[data.split.eq(split)].itertuples():
        name = Path(r.video).stem
        entry = {'video': name, 'animal_id': r.animal_id}
        try:
            for p in [r.video, r.pose_file, r.reference_mask]:
                if not Path(p).is_file():
                    raise FileNotFoundError(p)
            cap = cv2.VideoCapture(r.video)
            try:
                if not cap.isOpened():
                    raise ValueError('Unreadable video')
                entry['container_fps'] = cap.get(cv2.CAP_PROP_FPS)
                entry['acquisition_fps'] = r.fps
                entry['reported_frames'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            finally:
                cap.release()
            mask = np.load(r.reference_mask, allow_pickle=False)
            if not np.isin(mask, [0, 1]).all():
                raise ValueError('Expected binary reference mask')
            for key in ['video', 'pose_file', 'reference_mask']:
                entry[key + '_sha256'] = file_digest(getattr(r, key))
            windows = final_recording(r.video, r.pose_file, predictor, fish_class, output / name, r.fps, mask, parameters)
            windows['animal_id'] = r.animal_id
            results.append(windows)
            entry.update(status='processed', windows=len(windows), accepted=int(windows.bpm.notna().sum()))
        except Exception as exc:
            entry.update(status='failed', error=repr(exc))
        logs.append(entry)
        pd.DataFrame(logs).to_csv(output / 'deployment_log.csv', index=False)
    combined = pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=KEYS + ['animal_id', 'bpm'])
    combined.to_csv(output / 'deployment_windows.csv', index=False)
    return (combined, pd.DataFrame(logs))


### Manual cardiac annotations
Label the same cardiac phase on successive beats (e.g. ventricular contraction). Use half-open frame windows matching the prediction table. Record every planned reference window; do not omit failed automated measurements. Reference windows without enough visible beats should be logged separately as ungradable. The reference rate uses complete interbeat intervals, not the number of peaks divided by the window duration.

In [ ]:
def reference_from_beats(beats, windows):
    # beats: video, frame; windows: video, animal_id, start_frame, stop_frame, fps
    if windows.duplicated(KEYS).any() or beats.duplicated(['video','frame']).any():
        raise ValueError('Duplicate annotations')
    rows = []
    for r in windows.itertuples():
        if not np.isfinite(r.fps) or r.fps <= 0 or r.stop_frame <= r.start_frame:
            raise ValueError('Invalid reference window')
        f = np.sort(beats.loc[beats.video.eq(r.video) & beats.frame.ge(r.start_frame) & beats.frame.lt(r.stop_frame), 'frame'].to_numpy())
        rate = 60*r.fps*(len(f)-1)/(f[-1]-f[0]) if len(f)>=3 else np.nan
        rows.append(dict(video=r.video, animal_id=r.animal_id, start_frame=r.start_frame, stop_frame=r.stop_frame, reference_bpm=rate, annotated_beats=len(f)))
    return pd.DataFrame(rows)

# Example deployment (set paths and load the matching predictor first).
# pred, log = deploy_recordings(manifest_path, predictor, fish_class,
#     output_dir, frozen_parameters, checkpoint_path, split='test')
# ref = reference_from_beats(pd.read_csv(beat_path), pd.read_csv(window_path))
# ref.to_csv(output_dir / 'manual_reference.csv', index=False)
# metrics = independent_validation(pred, ref.dropna(subset=['reference_bpm']),
#     development_reference, output_dir / 'validation')


In [ ]:
# Check the beat-count calculation
_test_beats = pd.DataFrame({'video':['synthetic']*4,'frame':[0,100,200,300]})
_test_windows = pd.DataFrame([dict(video='synthetic',animal_id='test',start_frame=0,stop_frame=400,fps=300)])
assert np.isclose(reference_from_beats(_test_beats,_test_windows).reference_bpm.iloc[0],180)


### Experimental checks

Check that the predictor uses the checkpoint listed in the run configuration and that pose rows match decoded frames. The original training images must also be checked for overlap with test animals. The current runner needs precomputed pose files and one larva per recording.

## 20. Failure audit
Keep rejected and missing windows in the denominator. Status labels are diagnostic proxies, not confirmed biological causes.

In [ ]:
def failure_audit(pred, planned, log, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    if planned.duplicated(KEYS).any() or pred.duplicated(KEYS).any():
        raise ValueError('Duplicate windows')
    q = planned[KEYS + ['animal_id']].merge(pred.reindex(columns=KEYS + ['bpm', 'status']), on=KEYS, how='left', validate='one_to_one')
    failed = set(log.loc[log.status.eq('failed'), 'video'])
    q['outcome'] = np.where(q.bpm.notna(), 'accepted', q.status.fillna('missing_window'))
    q.loc[q.video.isin(failed) & q.bpm.isna(), 'outcome'] = 'recording_failed'
    counts = q.groupby(['video', 'outcome']).size().rename('windows').reset_index()
    counts['fraction'] = counts.windows / counts.groupby('video').windows.transform('sum')
    q.to_csv(folder / 'window_audit.csv', index=False)
    counts.to_csv(folder / 'failure_counts.csv', index=False)
    return (q, counts)


## 21. Performance and memory
Profile actual runs and inspect saved ROI arrays without loading them into RAM. The existing inference runner is unchanged; timings must be measured on the target hardware.

In [ ]:
import time
import tracemalloc

def profile_run(fn,*args,**kwargs):
    tracemalloc.start(); t=time.perf_counter()
    try:
        result=fn(*args,**kwargs)
        _,peak=tracemalloc.get_traced_memory()
        return result,{'elapsed_s':time.perf_counter()-t,'python_peak_mib':peak/2**20}
    finally:
        tracemalloc.stop()

def roi_inventory(path, chunk=256):
    if chunk<1: raise ValueError('Invalid chunk')
    a=np.load(path,mmap_mode='r',allow_pickle=False)
    if a.ndim!=3: raise ValueError('Expected frame × height × width')
    rows=[]
    for i in range(0,len(a),chunk):
        q=a[i:i+chunk]
        rows.append(pd.DataFrame({'frame':np.arange(i,i+len(q)), 'mean_intensity':q.mean(axis=(1,2)), 'std_intensity':q.std(axis=(1,2))}))
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame(columns=['frame','mean_intensity','std_intensity'])
# tracemalloc does not capture total native OpenCV or GPU memory.


## 22. Ablation studies
Supply predictions from separately executed variants on the same recordings. Use frozen parameters and identical planned reference windows. Compare fixed/adaptive ROI, harmonic screening and motion screening separately.

In [ ]:
def compare_variants(variants, reference, folder, missing_penalty=60.0):
    if reference.empty or reference.duplicated(KEYS).any():
        raise ValueError('Invalid reference')
    if not np.isfinite(reference.reference_bpm).all() or (reference.reference_bpm <= 0).any():
        raise ValueError('Invalid reference BPM')
    if not variants:
        raise ValueError('No variants')
    pairs = []
    rows = []
    for name, pred in variants.items():
        q = reference.merge(pred[KEYS + ['bpm']], on=KEYS, how='left', validate='one_to_one')
        q['bpm'] = q.bpm.where(np.isfinite(q.bpm) & q.bpm.gt(0))
        q['error'] = q.bpm - q.reference_bpm
        q['variant'] = name
        e = q.error
        rows.append(dict(variant=name, coverage=e.notna().mean(), mae=e.abs().mean(), rmse=np.sqrt((e ** 2).mean()), bias=e.mean(), loss=e.abs().fillna(missing_penalty).mean(), matched=e.notna().sum(), planned=len(q)))
        pairs.append(q)
    pairs = pd.concat(pairs, ignore_index=True)
    summary = pd.DataFrame(rows)
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    pairs.to_csv(folder / 'ablation_pairs.csv', index=False)
    summary.to_csv(folder / 'ablation_summary.csv', index=False)
    return (summary, pairs)


## 23. Results figures
Generate figures only from supplied experimental predictions and manual reference measurements. Agreement limits are pooled descriptive statistics; animal-level bootstrap intervals are available in Stage 17.

In [ ]:
import matplotlib.pyplot as plt

def research_figures(pairs, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    q = pairs.dropna(subset=['bpm', 'reference_bpm']).copy()
    if len(q) < 2:
        raise ValueError('At least two matched windows required')
    q['error'] = q.bpm - q.reference_bpm
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(q.reference_bpm, q.bpm, s=15, alpha=0.65)
    lo = min(q.reference_bpm.min(), q.bpm.min())
    hi = max(q.reference_bpm.max(), q.bpm.max())
    ax.plot([lo, hi], [lo, hi], linestyle='--')
    ax.set(xlabel='Manual reference (BPM)', ylabel='Automated estimate (BPM)')
    fig.tight_layout()
    fig.savefig(folder / 'agreement.pdf')
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter((q.bpm + q.reference_bpm) / 2, q.error, s=15, alpha=0.65)
    bias = q.error.mean()
    sd = q.error.std(ddof=1)
    for y in [bias, bias - 1.96 * sd, bias + 1.96 * sd]:
        ax.axhline(y, linestyle='--')
    ax.set(xlabel='Mean of methods (BPM)', ylabel='Automated − manual (BPM)')
    fig.tight_layout()
    fig.savefig(folder / 'bland_altman.pdf')
    plt.close(fig)
    return {'matched': len(q), 'reference_windows': len(pairs), 'coverage': len(q) / len(pairs)}


## 24. Research release
Archive outputs, configuration and software versions. Keep raw video and model checkpoints separately under the applicable data permissions. Review files before sharing; manifests can contain private paths.

In [ ]:
import platform
import importlib.metadata as metadata
import zipfile

def release_results(run_folder, archive):
    run_folder = Path(run_folder).resolve()
    archive = Path(archive).resolve()
    if not run_folder.is_dir():
        raise FileNotFoundError(run_folder)
    if archive.is_relative_to(run_folder):
        raise ValueError('Archive must be outside run folder')
    versions = {}
    for name in ['numpy', 'pandas', 'scipy', 'opencv-python', 'torch', 'detectron2', 'deeplabcut', 'matplotlib']:
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    files = [p for p in sorted(run_folder.rglob('*')) if p.is_file() and (not p.is_symlink()) and (p.suffix.lower() in {'.csv', '.json', '.pdf', '.png'})]
    if not files:
        raise ValueError('No results to archive')
    manifest = {'python': platform.python_version(), 'packages': versions, 'files': {str(p.relative_to(run_folder)): file_digest(p) for p in files}}
    archive.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, 'results/' + str(p.relative_to(run_folder)))
        z.writestr('release_manifest.json', json.dumps(manifest, indent=2))
    return archive


### Order of the experiments

1. Check camera metadata, model files and animal IDs.
2. Process the development recordings and choose parameters.
3. Annotate beats independently and define the reference windows.
4. Compare analysis variants and inspect failed windows.
5. Freeze settings, process the test recordings and calculate agreement and coverage.
6. Save the figures, tables and settings.

Experimental accuracy and processing speed are not yet measured. Simultaneous BPM for several larvae is not implemented.

In [ ]:
# Checks using simulated data (not fish recordings)
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    root=Path(tmp)
    ref=pd.DataFrame({'video':['sim']*2,'animal_id':['a']*2,'start_frame':[0,100],'stop_frame':[100,200],'reference_bpm':[180.,180.]})
    pred=ref.iloc[:1][KEYS].assign(bpm=175.,status='accepted')
    log=pd.DataFrame({'video':['sim'],'status':['processed']})
    q,c=failure_audit(pred,ref,log,root)
    assert q.outcome.tolist()==['accepted','missing_window']
    s,p=compare_variants({'baseline':pred},ref,root)
    assert s.coverage.iloc[0]==.5 and s.mae.iloc[0]==5
    np.save(root/'roi.npy',np.ones((5,8,8)))
    assert len(roi_inventory(root/'roi.npy',2))==5
    result,profile=profile_run(sum,[1,2,3]); assert result==6
    p.loc[p.bpm.isna(),'bpm']=180
    assert research_figures(p,root)['matched']==2
    assert release_results(root,root.parent/(root.name+'.zip')).is_file()
    (root.parent/(root.name+'.zip')).unlink()
print('Simulated analysis checks passed')